# Google AI 工作坊：AI 精實創業實戰 (AutoGen 框架)

本教學筆記本透過 **AutoGen** 實作「AI 精實創業框架」，將原本需要手動切換的 6 個 AI 角色串聯成一個自動化的工作流。
核心理念：**先賣概念，再做產品** — 用 AI 把「根本還沒生產出來的實體硬體或日用品」逼真地畫出來，上線預購測試，有人買單才去談工廠！

我們將走過「痛點盲測 ➔ AI 產品概念圖 ➔ 零代碼預購網頁上線」三個階段。

## AI 夢幻團隊 (Agent Team):
1. **首席市場調查顧問**：挖掘目標客群未被滿足的剛需，提出全新實體產品開發概念
2. **矽谷嚴苛創投 VC**：針對技術可行性、供應鏈 MOQ 風險、預購信任風險進行壓力測試
3. **首席產品經理 PM**：將痛點轉化為產品規格書 (PRD)，定調核心功能與視覺材質氛圍
4. **視覺與特效大導**：撰寫給 Nano Banana 2 (Pro) 的終極產品概念圖提示詞
5. **全端 UI/UX 首席工程師**：將 PRD 轉譯成 Vibe Coding 架站提示詞（預購登陸頁）
6. **動態與音效大師**：撰寫給 Veo (影片) 與 Lyria 3 (配樂) 的精確指令

## 工具鏈：
Gemini、Gemini Gems、Firebase (Vibe Coding)、Nano Banana 2 (產品渲染)、Veo & Lyria 3 (動態展示與配樂)

In [1]:
# 安裝 AutoGen (取消註解並執行)
!pip install pyautogen

## 1. 設定環境與 LLM 引擎
請設定你的 API Key。這套框架可套用在 OpenAI、Google Gemini 或其他模型上。

In [1]:
import autogen
import os

# ── 共用 config list（所有 Agent 使用相同模型，各自有不同 temperature）──
_config_list = [{"model": "gpt-4o", "api_key": os.environ.get("OPENAI_API_KEY")}]

# 🔍 首席市場調查顧問：分析型，需精準 → 低溫
llm_config_market   = {"config_list": _config_list, "temperature": 0.3}

# 💰 矽谷嚴苛創投 VC：邏輯壓力測試，需犀利一致 → 最低溫
llm_config_vc       = {"config_list": _config_list, "temperature": 0}

# 📋 首席產品經理 PM：需求定調，需穩健精確 → 中低溫
llm_config_pm       = {"config_list": _config_list, "temperature": 0.3}

# 🎬 視覺特效大導：AI 提示詞需精準且有創意 → 中高溫
llm_config_director = {"config_list": _config_list, "temperature": 0.8}

# 💻 首席工程師：技術輸出需精確 → 最低溫
llm_config_engineer = {"config_list": _config_list, "temperature": 0}

# 🎵 動態與音效大師：影音提示詞需天馬行空 → 高溫
llm_config_av_master = {"config_list": _config_list, "temperature": 1.0}

## 2. 建立 AI 夢幻團隊 (定義 Agents)
根據課程大綱，我們將所有 Gem 系統指令轉換成 AutoGen 代理人的 。
流程對應三大階段：
- **第一階段（現場盲測與大腦激盪）**：市場顧問 → VC → PM
- **第二階段（AI 視覺具象化與 Vibe Coding 架站）**：視覺大導 → 工程師
- **第三階段（影音動態展示與收斂）**：動態與音效大師

In [7]:
# ── 1. 首席市場調查顧問（temperature=0.3：精準分析）──
market_advisor = autogen.AssistantAgent(
    name="Market_Advisor",
    system_message=(
        "你是一間頂尖管顧公司的合夥人，專精於 D2C 實體產品品牌與精實創業模式。"
        "你的任務是找出目標客群『未被滿足的剛需』，並提出全新的實體硬體或日用品開發概念。\n"
        "當我給你一個客群時，你必須：\n"
        "1. 精準點出他們在日常生活中最常遇到的三個具體挫折與痛點。\n"
        "2. 針對這三個痛點，大膽提出三個全新的實體產品開發概念（硬體或日用品，非軟體）。\n"
        "3. 分析每個概念為何具備讓消費者『現在就想掏錢預購』的強烈動機。\n"
        "你的提案必須拒絕紅海市場的平庸產品，專攻有明確痛點且消費者願意付溢價的利基市場。"
        "語氣必須犀利、直指商機核心。"
    ),
    llm_config=llm_config_market,
)

# ── 2. 矽谷嚴苛創投 VC（temperature=0：邏輯犀利、一致性高）──
vc = autogen.AssistantAgent(
    name="Silicon_Valley_VC",
    system_message=(
        "你是一位極度挑剔、看過無數硬體新創死在沙灘上的頂級創投合夥人。"
        "我會向你簡報一個「先預購收單、確認需求後再找工廠生產」的實體產品計畫。\n"
        "你的任務是無情攻擊我，不准說任何客套話。\n"
        "請針對以下三個核心維度對我進行靈魂拷問：\n"
        "1. 技術可行性：這個產品的核心技術真的做得出來嗎？有沒有物理定律或工程瓶頸會讓它變成空中樓閣？\n"
        "2. 供應鏈與最低起訂量 (MOQ) 風險：開模費用多少？最低起訂量是多少？小量預購訂單撐得起第一批生產嗎？\n"
        "3. 消費者預購信任風險：產品根本還沒做出來，消費者憑什麼相信你？你要如何解決「收了錢卻交不出貨」的信任危機？\n"
        "逼我講出避險策略與應對方案。"
    ),
    llm_config=llm_config_vc,
)

# ── 3. 首席產品經理 PM（temperature=0.3：需求定調精準）──
pm = autogen.AssistantAgent(
    name="Product_Manager",
    system_message=(
        "你是一位擁有豐富硬體產品開發經驗的首席產品經理。"
        "我會提供你產品概念與創投 (VC) 的質疑。\n"
        "你的任務是將模糊的產品概念轉化為清晰的產品規格，並回應 VC 的質疑。請產出以下內容：\n"
        "1. 三大核心功能亮點：針對 VC 對『技術可行性與信任感』的質疑，定義三個最能打動消費者的功能賣點。\n"
        "2. 視覺與材質氛圍定調：明確指定產品的視覺風格與材質感受（例如：科技感金屬、溫潤木質、高階消光材質等）。\n"
        "3. 產品需求規格書 (PRD)：將上述內容總結成一份條理分明的 PRD，包含產品名稱、目標客群、核心功能、材質規格、視覺定調，讓後續的視覺大導與工程師能直接使用。"
    ),
    llm_config=llm_config_pm,
)

# ── 4. 視覺與特效大導（temperature=0.8：AI 提示詞需精準有創意）──
director = autogen.AssistantAgent(
    name="Director",
    system_message=(
        "你是精通高階產品商攝 (Product Photography) 與概念渲染的國際大導演。"
        "現在我們連一塊材料都還沒買，但要生出產品的頂級商攝概念圖！\n"
        "當我給你一份產品規格 (PRD) 時，請幫我寫出一段給 Nano Banana 2 的終極英文提示詞 (Prompt)。\n"
        "這段提示詞必須：\n"
        "1. 精準描述產品材質的微觀質感（如金屬拉絲紋理、木紋肌理、消光塗層的觸感）。\n"
        "2. 詳細描述產品的結構設計與外觀輪廓。\n"
        "3. 設定符合產品氛圍的專業光影（如 Cinematic lighting、Studio lighting with dramatic shadows、Soft diffused natural light）。\n"
        "4. 指定攝影機參數（如 85mm lens, f/2.8, product photography on clean background）。\n"
        "請直接給我這段完整的英文 Prompt，不要有任何中文或廢話。\n"
        "提醒：永遠建議使用者切換到 Pro 模型 (Redo with Pro) 以獲得最佳材質光澤與紋理！"
    ),
    llm_config=llm_config_director,
)

# ── 5. 全端與 UI/UX 首席工程師（temperature=0：技術精確）──
engineer = autogen.AssistantAgent(
    name="Chief_Engineer",
    system_message=(
        "你是曾待過蘋果與 Google、深諳現代網頁美學的頂級全端開發者兼 UI/UX 總監。"
        "我會給你產品經理 (PM) 剛開出的『產品需求規格書 (PRD)』。\n"
        "你的任務是將這些需求轉譯成一段可以丟給 Vibe Coding 工具 (如 Google AI Studio + Firebase) 的『完美架站提示詞』。\n"
        "目標：建立一個高轉換率的預購登陸頁 (Pre-order Landing Page)。\n"
        "你的產出必須是一段可以直接複製貼上的完整指令，結構包含：\n"
        "1. 技術與框架指定：HTML/Tailwind CSS，完全響應式 (RWD)。\n"
        "2. 視覺風格定義：根據產品調性指定主色調、字體風格、UI 風格。\n"
        "3. 網頁結構（由上到下）：\n"
        "   - Hero Section（滿版產品概念圖、震撼大標題、副標語）\n"
        "   - 三大亮點圖文區塊（對應 PM 定義的三大功能賣點）\n"
        "   - 『加入等候名單 (Waitlist) 解鎖早鳥優惠』Email 收集表單（這是最重要的轉換元素！）\n"
        "   - Footer\n"
        "4. 佔位符設定：圖片預留高質感佔位圖 (Placeholder)。\n"
        "請不要有任何廢話，直接給予那段完整的架站咒語。"
    ),
    llm_config=llm_config_engineer,
)

# ── 6. 動態與音效大師（temperature=1.0：影音提示詞天馬行空）──
av_master = autogen.AssistantAgent(
    name="AV_Master",
    system_message=(
        "你是精通動態影像與音效設計的國際級廣告導演兼音效總監。"
        "我們已有產品的靜態概念圖，現在要讓它動起來並配上專業級音樂！\n"
        "當我給你產品資訊時，請給我兩段明確的英文指令：\n"
        "1. 給影片 AI (Veo) 的指令：\n"
        "   - 描述運鏡方式（如 Slow tracking shot、Orbiting camera movement）\n"
        "   - 展示產品特有的微小物理動態（如材質反光變化、燈號閃爍、機械結構運作、液體流動等）\n"
        "   - 務必讓動態真實自然，擺脫 AI 塑膠感\n"
        "2. 給音樂 AI (Lyria 3) 的指令：\n"
        "   - 描述 30 秒背景配樂的曲風與 BPM\n"
        "   - 指定適合的樂器組合\n"
        "   - 營造完美符合產品氛圍的情緒（如高階科技感、溫馨日常感、專業工藝感等）\n"
        "請全部以英文輸出，語法必須像指導頂級特效團隊一樣專業。\n"
        "提醒：AI 生成的音樂皆帶有 SynthID 水印保護機制。"
    ),
    llm_config=llm_config_av_master,
)

# 代表 User（人類）負責發起對話與中繼
# human_input_mode 選項：
#   "NEVER"  = 全自動（適合跑完整 pipeline，不需人工介入）
#   "ALWAYS" = 每次 Agent 回覆後都等你手動輸入（適合工作坊現場 Demo，可以挑選產品概念）
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="ALWAYS",       # 工作坊現場改成 "ALWAYS" 可手動輸入
    max_consecutive_auto_reply=0,    # ⚠️ 設為 0：每個 Agent 只回答一次就結束，避免空訊息循環
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

## 2.5 🎛️ Agent 排程控制台 (GUI 排序 + 迴圈設定)

執行此 Cell 會顯示互動式控制台：
- **▲ / ▼ 按鈕**：拖動 Agent 改變說話順序
- **🔁 Loop 核取框**：勾選後，勾選的相連 Agent 會組成一個迴圈群組
- **迴圈次數**：設定迴圈群組要重複幾輪
- **▶ Run Pipeline 按鈕**：依目前排序與迴圈設定執行工作流

### 大綱對應流程：
1. 第一階段（0:10-0:40）：首席市場調查顧問 → 矽谷嚴苛創投 VC → 首席產品經理 PM
2. 第二階段（0:40-1:15）：視覺與特效大導 → 全端工程師
3. 第三階段（1:15-1:40）：動態與音效大師

In [ ]:
# ============================================================
# Cell 2.5-A  Agent 排程 GUI 控制台
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

# ═══════════════════════════════════════════════════════════════
# 🎯 在這裡設定你的目標客群（現場 Demo 時直接改這行！）
# ═══════════════════════════════════════════════════════════════
target_audience = "想穿起來有型但又需要拿來登山的人"

# ----- agent registry -----------------------------------------------
AGENT_REGISTRY = {
    "Market_Advisor":    (market_advisor, "首席市場調查顧問", "🔍"),
    "Silicon_Valley_VC": (vc,            "矽谷嚴苛創投 VC",  "💰"),
    "Product_Manager":   (pm,            "首席產品經理 PM",   "📋"),
    "Director":          (director,      "視覺特效大導",      "🎬"),
    "Chief_Engineer":    (engineer,      "全端工程師",         "🛠️"),
    "AV_Master":         (av_master,     "動態與音效大師",    "🎵"),
}

# Default messages each agent receives in the pipeline
# 注意：Market_Advisor 的訊息使用 {target_audience} 佔位符，
#       會在 run_pipeline() 中動態替換
AGENT_MESSAGES = {
    # 🎤 第一階段：現場盲測與大腦激盪 (0:10 - 0:40)
    "Market_Advisor": (
        "我現在想創立一個全新的原生硬體/日用品品牌。"
        "請幫我分析『{target_audience}』在日常生活中，最常遇到的三個具體挫折。"
        "針對這三個痛點，大膽提出三個全新的實體產品開發概念。"
        "確保這個缺口具備讓消費者『現在就想掏錢預購』的強烈動機。"
    ),
    "Silicon_Valley_VC": (
        "這是我剛才跟顧問討論出來的產品概念（請參考上方市場顧問的輸出）。"
        "我們打算先做預購網頁收單，確認需求後再找工廠生產。"
        "請以嚴苛創投的角度，針對『技術可行性』、『供應鏈與最低起訂量 (MOQ) 風險』"
        "以及『消費者預購信任風險』這三個維度直接打臉我，不准說客套話。"
    ),
    "Product_Manager": (
        "PM 你好，我們決定開發上述產品概念（請參考前面的討論）。"
        "為了回應剛剛 VC 對我們『技術可行性與信任感』的質疑，"
        "請幫我寫出這款產品的『三大核心功能亮點』，"
        "並定調它的『視覺與材質氛圍』（例如：科技感金屬、溫潤木質、或是高階消光材質等）。"
        "最後總結成一份產品需求規格書 (PRD)。"
    ),
    # 🎤 第二階段：AI 視覺具象化與 Vibe Coding 架站 (0:40 - 1:15)
    "Director": (
        "導演，這是 PM 剛定義好的產品規格（請參考上方 PM 的 PRD 輸出）。"
        "現在我要使用 Nano Banana 2 生成產品概念圖。"
        "請幫我寫出終極的英文提示詞 (Prompt)，"
        "精準描述產品材質的微觀質感、結構設計，"
        "以及符合產品氛圍的光影設定。請直接給我這段完整的英文 Prompt。"
    ),
    "Chief_Engineer": (
        "工程師，請幫我把這份需求轉譯成 Vibe Coding 架站提示詞（請參考上方 PM 的 PRD）。"
        "目標是建立一個高轉換率的預購登陸頁。"
        "請指定使用 HTML/Tailwind CSS，視覺風格請根據產品調性設定。"
        "必須包含 Hero Section、三大亮點圖文區塊，"
        "以及最重要的『加入等候名單 (Waitlist) 解鎖早鳥優惠』Email 收集表單。"
        "直接給我完整的架站提示詞。"
    ),
    # 🎤 第三階段：影音動態展示與收斂 (1:15 - 1:40)
    "AV_Master": (
        "大師，產品的靜態概念圖已經生成。請給我兩段明確的英文指令：\n"
        "1. 給影片 AI (Veo) 的指令：描述 Slow tracking shot 運鏡，"
        "並展示產品特有的微小物理動態（如材質反光變化、燈號閃爍或機械結構運作）。\n"
        "2. 給音樂 AI (Lyria 3) 的指令：描述 30 秒背景配樂，"
        "指定適合的曲風與樂器，營造完美符合產品氛圍的情緒。"
    ),
}

# ----- mutable state -------------------------------------------------
agent_stack = list(AGENT_REGISTRY.keys())    # ordered list of agent keys
loop_flags  = {k: False for k in agent_stack}  # which agents are in loop
loop_iterations = 2                           # default loop rounds

# ----- card styles ---------------------------------------------------
CARD_LAYOUT      = widgets.Layout(border="2px solid #4A90D9", border_radius="10px",
                                  padding="6px 12px", margin="3px 0")
LOOP_CARD_LAYOUT = widgets.Layout(border="2px solid #e94560", border_radius="10px",
                                  padding="6px 12px", margin="3px 0")

out = widgets.Output()


def build_ui():
    """Render the full control panel from current agent_stack state."""
    with out:
        clear_output(wait=True)

        title = widgets.HTML(
            value="<h3 style='color:#4A90D9;font-family:monospace'>"
                  "🎛️ Agent 排程控制台</h3>"
        )

        rows = []
        for idx, key in enumerate(agent_stack):
            _, label, emoji = AGENT_REGISTRY[key]

            badge = widgets.HTML(
                value=f"<span style='color:#aaa;font-size:12px;width:24px;display:inline-block'>#{idx+1}</span>"
            )

            btn_up = widgets.Button(
                description="▲",
                layout=widgets.Layout(width="36px", height="28px"),
                style=widgets.ButtonStyle(button_color="#2c3e50")
            )
            btn_dn = widgets.Button(
                description="▼",
                layout=widgets.Layout(width="36px", height="28px"),
                style=widgets.ButtonStyle(button_color="#2c3e50")
            )
            btn_up.disabled = (idx == 0)
            btn_dn.disabled = (idx == len(agent_stack) - 1)

            def make_move(i, direction):
                def _move(b):
                    j = i + direction
                    agent_stack[i], agent_stack[j] = agent_stack[j], agent_stack[i]
                    build_ui()
                return _move

            btn_up.on_click(make_move(idx, -1))
            btn_dn.on_click(make_move(idx, +1))

            lbl = widgets.HTML(
                value=f"<span style='color:#e0e0e0;font-family:monospace;font-size:14px'>"
                      f"{emoji} {label}</span>",
                layout=widgets.Layout(width="260px")
            )

            chk = widgets.ToggleButton(
                value=loop_flags[key],
                description="🔁 Loop",
                button_style="danger" if loop_flags[key] else "",
                layout=widgets.Layout(width="90px", height="28px"),
            )

            def make_toggle(k):
                def _toggle(change):
                    loop_flags[k] = change["new"]
                    build_ui()
                return _toggle

            chk.observe(make_toggle(key), names="value")

            card_layout = LOOP_CARD_LAYOUT if loop_flags[key] else CARD_LAYOUT
            row = widgets.HBox([badge, btn_up, btn_dn, lbl, chk], layout=card_layout)
            rows.append(row)

        any_loop = any(loop_flags.values())
        loop_section = widgets.HBox([])
        if any_loop:
            iter_label = widgets.HTML(
                value="<span style='color:#e94560;font-family:monospace'>🔁 迴圈次數：</span>"
            )
            spin = widgets.BoundedIntText(
                value=loop_iterations, min=1, max=20, step=1,
                layout=widgets.Layout(width="80px"),
                style={"description_width": "0px"},
            )

            def on_iter_change(change):
                global loop_iterations
                loop_iterations = change["new"]

            spin.observe(on_iter_change, names="value")
            loop_section = widgets.HBox(
                [iter_label, spin],
                layout=widgets.Layout(margin="6px 0", padding="6px 12px",
                                      border="1px dashed #e94560",
                                      border_radius="6px")
            )

        preview_parts = []
        i = 0
        while i < len(agent_stack):
            k = agent_stack[i]
            _, lbl_str, em = AGENT_REGISTRY[k]
            if loop_flags[k]:
                group_labels = []
                while i < len(agent_stack) and loop_flags[agent_stack[i]]:
                    _, ls, le = AGENT_REGISTRY[agent_stack[i]]
                    group_labels.append(f"{le}{ls}")
                    i += 1
                inner = " ↔ ".join(group_labels)
                preview_parts.append(
                    f"<span style='color:#e94560'>[🔁 {inner} ×{loop_iterations}]</span>"
                )
            else:
                preview_parts.append(
                    f"<span style='color:#4A90D9'>{em}{lbl_str}</span>"
                )
                i += 1

        preview_html = " ➔ ".join(preview_parts)
        preview = widgets.HTML(
            value=f"<div style='font-size:12px;font-family:monospace;margin-top:8px;padding:6px 12px;background:#0d0d1a;border-radius:6px'>"
                  f"<b style='color:#888'>流程預覽：</b> {preview_html}</div>"
        )

        stack_box = widgets.VBox(rows, layout=widgets.Layout(margin="8px 0"))
        display(widgets.VBox([title, stack_box, loop_section, preview]))


build_ui()
out

Output()

### 🚀 執行 Pipeline
設定好上面的排序與迴圈後，在下方輸入主題並執行 Cell 即可啟動。


In [9]:
# ============================================================
# Cell 2.5-B  依 GUI 設定執行 Pipeline（含迴圈群組支援）
# ============================================================

def run_pipeline(topic: str):
    """
    Execute the agent pipeline in the order defined by the GUI.

    Parameters
    ----------
    topic : str
        描述目標客群，會被填入 Market_Advisor 的 {target_audience} 佔位符。
        同時也作為整條 pipeline 的主題。
    """
    chat_sequence = []
    i = 0

    while i < len(agent_stack):
        key = agent_stack[i]
        agent_obj, _, _ = AGENT_REGISTRY[key]

        if loop_flags[key]:
            # ── Collect all consecutive loop-enabled agents ──────
            loop_group = []
            while i < len(agent_stack) and loop_flags[agent_stack[i]]:
                k = agent_stack[i]
                loop_group.append((k, AGENT_REGISTRY[k][0]))
                i += 1

            # ── Unroll rounds ────────────────────────────────────
            names = [g[0] for g in loop_group]
            print(f"🔁 Loop group: {names} × {loop_iterations} rounds")
            for round_num in range(loop_iterations):
                for k, loop_agent_obj in loop_group:
                    base_msg = AGENT_MESSAGES.get(k, "請繼續上一輪的討論，給出新的見解。")
                    # 動態填入 target_audience
                    base_msg = base_msg.format(target_audience=topic)
                    if round_num > 0:
                        base_msg = f"(迴圈第 {round_num + 1} 輪) {base_msg}"
                    chat_sequence.append({
                        "recipient": loop_agent_obj,
                        "message": base_msg,
                        "summary_method": "last_msg",
                    })
        else:
            msg = AGENT_MESSAGES.get(key, "請依據前面討論提供你的專業建議。")
            # 動態填入 target_audience
            msg = msg.format(target_audience=topic)
            chat_sequence.append({
                "recipient": agent_obj,
                "message": msg,
                "summary_method": "last_msg",
            })
            i += 1

    print(f"\n▶ 開始執行 Pipeline，共 {len(chat_sequence)} 個對話步驟…\n")
    results = user_proxy.initiate_chats(chat_sequence)
    print("\n" + "="*50 + "\n🎉 Pipeline 執行完畢 🎉\n" + "="*50)
    for idx, res in enumerate(results):
        print(f"\n--- 步驟 {idx+1} 成果 ---")
        print(res.summary)
    return results


# ── 執行：修改 target_audience 後執行此 Cell ──────────────────────────
# 💡 這裡的 target_audience 就是你在台上問學員得到的客群！
results = run_pipeline(target_audience)


▶ 開始執行 Pipeline，共 6 個對話步驟…


********************************************************************************
Starting a new chat....

********************************************************************************
User_Proxy (to Market_Advisor):

我現在想創立一個全新的原生硬體/日用品品牌。請幫我分析『想穿起來有型但又需要拿來登山的人』在日常生活中，最常遇到的三個具體挫折。針對這三個痛點，大膽提出三個全新的實體產品開發概念。確保這個缺口具備讓消費者『現在就想掏錢預購』的強烈動機。

--------------------------------------------------------------------------------
Market_Advisor (to User_Proxy):

目標客群：想穿起來有型但又需要拿來登山的人

### 三個具體挫折與痛點

1. **功能與時尚的矛盾**：
   - 許多戶外服裝在功能性上表現出色，但外觀設計往往過於單調或缺乏時尚感，讓消費者在城市與山野之間切換時感到不自在。

2. **多變天氣的適應性**：
   - 登山時，天氣變化無常，從晴天到暴雨，溫差大，現有的服裝難以快速適應這些變化，消費者需要攜帶多件衣物，增加負擔。

3. **舒適度與耐用性的平衡**：
   - 許多戶外服裝在耐用性上表現出色，但舒適度往往被犧牲，長時間穿著容易產生不適，影響整體登山體驗。

### 三個全新實體產品開發概念

1. **變形時尚戶外夾克**：
   - **概念**：一款能夠根據環境變化自動調節的時尚夾克，採用智能材料技術，能夠在不同溫度和濕度下自動改變保暖性和透氣性，同時具備可拆卸的時尚外罩，讓使用者在城市和山野之間自由切換。
   - **強烈動機**：消費者不再需要在功能和時尚之間妥協，這款夾克提供了兩全其美的解決方案，讓他們在任何環境下都能保持最佳狀態。

2. **全天候智能登山鞋**：
   - **概念**：這款鞋子內置微型氣候感應器，能夠根據天

## 3. 執行自動化工作流 (Agent Orchestration)
我們透過 AutoGen 的 `initiate_chats` 來串接工作流。資訊會自動在代理人之間傳遞：

### 第一階段：現場盲測與大腦激盪
1. User ➔ Market Advisor (挖掘痛點、提出產品概念)
2. Advisor ➔ VC (技術可行性、供應鏈、預購信任壓力測試)
3. VC ➔ PM (定義核心功能、視覺材質氛圍、產出 PRD)

### 第二階段：AI 視覺具象化與 Vibe Coding 架站
4. PM ➔ 視覺大導 (產出 Nano Banana 2 Pro 概念圖提示詞)
5. PM ➔ 工程師 (產出預購登陸頁 Vibe Coding 提示詞)

### 第三階段：影音動態展示
6. 大導 ➔ 動態與音效大師 (產出 Veo 影片 + Lyria 3 配樂指令)

In [5]:
# ═══════════════════════════════════════════════════════════════
# 直接執行版（不經過 GUI，適合快速測試）
# ═══════════════════════════════════════════════════════════════

# 啟動對話鏈
chat_results = user_proxy.initiate_chats([
    # ═══ 第一階段：現場盲測與大腦激盪 ═══
    {
        "recipient": market_advisor,
        "message": (
            f"我現在想創立一個全新的原生硬體/日用品品牌。"
            f"請幫我分析『{target_audience}』在日常生活中，"
            f"最常遇到的三個具體挫折。針對這三個痛點，"
            f"大膽提出三個全新的實體產品開發概念。"
            f"確保這個缺口具備讓消費者『現在就想掏錢預購』的強烈動機。"
        ),
        "summary_method": "last_msg",
    },
    # 階段二：矽谷嚴苛創投壓力測試
    {
        "recipient": vc,
        "message": (
            "這是我剛才跟顧問討論出來的產品概念。"
            "我們打算先做預購網頁收單，確認需求後再找工廠生產。"
            "請以嚴苛創投的角度，針對『技術可行性』、"
            "『供應鏈與最低起訂量 (MOQ) 風險』以及"
            "『消費者預購信任風險』這三個維度直接打臉我，不准說客套話。"
        ),
        "summary_method": "last_msg",
    },
    # 階段三：首席產品經理定調產品規格
    {
        "recipient": pm,
        "message": (
            "PM 你好，我們決定開發上述產品概念。"
            "為了回應剛剛 VC 對我們『技術可行性與信任感』的質疑，"
            "請幫我寫出這款產品的『三大核心功能亮點』，"
            "並定調它的『視覺與材質氛圍』（例如：科技感金屬、溫潤木質、或是高階消光材質等）。"
            "最後總結成一份產品需求規格書 (PRD)。"
        ),
        "summary_method": "reflection_with_llm",
    },
    # ═══ 第二階段：AI 視覺具象化與 Vibe Coding 架站 ═══
    {
        "recipient": director,
        "message": (
            "導演，這是 PM 剛定義好的產品規格。"
            "現在我要使用 Nano Banana 2 生成產品概念圖。"
            "請幫我寫出終極的英文提示詞 (Prompt)，"
            "精準描述產品材質的微觀質感、結構設計，"
            "以及符合產品氛圍的光影設定。請直接給我這段完整的英文 Prompt。"
        ),
        "summary_method": "last_msg",
    },
    {
        "recipient": engineer,
        "message": (
            "工程師，請幫我把 PM 的產品需求規格書轉譯成 Vibe Coding 架站提示詞。"
            "目標是建立一個高轉換率的預購登陸頁。"
            "請指定使用 HTML/Tailwind CSS，視覺風格請根據產品調性設定。"
            "必須包含 Hero Section、三大亮點圖文區塊，"
            "以及最重要的『加入等候名單 (Waitlist) 解鎖早鳥優惠』Email 收集表單。"
            "直接給我完整的架站提示詞。"
        ),
        "summary_method": "last_msg",
    },
    # ═══ 第三階段：影音動態展示與收斂 ═══
    {
        "recipient": av_master,
        "message": (
            "大師，產品的靜態概念圖已經生成。請給我兩段明確的英文指令：\n"
            "1. 給影片 AI (Veo) 的指令：描述 Slow tracking shot 運鏡，"
            "並展示產品特有的微小物理動態（如材質反光變化、燈號閃爍或機械結構運作）。\n"
            "2. 給音樂 AI (Lyria 3) 的指令：描述 30 秒背景配樂，"
            "指定適合的曲風與樂器，營造完美符合產品氛圍的情緒。"
        ),
        "summary_method": "last_msg",
    },
])


********************************************************************************
Starting a new chat....

********************************************************************************
User_Proxy (to Market_Advisor):

我現在想創立一個全新的原生硬體/日用品品牌。請幫我分析『想穿起來有型但又需要拿來登山的人』在日常生活中，最常遇到的三個具體挫折。針對這三個痛點，大膽提出三個全新的實體產品開發概念。確保這個缺口具備讓消費者『現在就想掏錢預購』的強烈動機。

--------------------------------------------------------------------------------
Market_Advisor (to User_Proxy):

目標客群：想穿起來有型但又需要拿來登山的人

### 痛點分析

1. **功能與時尚的衝突**：
   - 許多戶外裝備在功能性上表現出色，但設計上往往顯得笨重或不夠時尚，無法滿足消費者在都市與戶外之間無縫切換的需求。

2. **多變天氣的挑戰**：
   - 登山時，天氣變化無常，消費者需要能夠快速適應不同天氣條件的服裝，但現有產品往往需要攜帶多件衣物，增加了負擔。

3. **舒適性與耐用性的平衡**：
   - 許多時尚的登山服裝在舒適性上有所欠缺，或者耐用性不足，無法應對長時間的戶外活動。

### 產品開發概念

1. **變形時尚登山夾克**：
   - **概念**：一款可變形的夾克，採用模組化設計，能夠根據不同場合和天氣條件進行調整。夾克的袖子、內襯和帽子可以快速拆卸或更換，從而在都市和山野間自由切換。
   - **動機**：消費者渴望一件能夠在任何環境下都保持時尚的單品，這款夾克不僅滿足了功能需求，還能讓他們在城市中也能穿出風格。

2. **智能氣候調節登山鞋**：
   - **概念**：這款鞋子內置智能氣候調節系統，能夠根據外部溫度和濕度自動調整鞋內環境，保持雙腳的乾爽和舒適。鞋面設計時尚，適合日常穿著。
   - **動機**：消費者不再需要

## 4. 取得成果
工作流執行完畢後，你已經獲得了：
1. ✅ 經過市場驗證的產品概念（附痛點分析）
2. ✅ 通過創投壓力測試的避險策略
3. ✅ 完整的產品需求規格書 (PRD)
4. ✅ Nano Banana 2 Pro 級產品概念圖提示詞（頂級商攝等級）
5. ✅ Vibe Coding 預購登陸頁架站咒語（可直接丟入 Google AI Studio + Firebase）
6. ✅ Veo 影片 + Lyria 3 配樂的英文指令（動態展示素材）

> 💡 **下一步**：如果你的預購網頁收到了 500 個 Email，這就是一個被真實市場驗證的剛需！
> 帶著這 500 張意向訂單去跟工廠談判開模。用極低的成本試錯，用真實的數據決定生產！

In [7]:
# 印出每個階段的最終結果 (可選)
print("\n" + "="*50 + "\n🎉 所有階段執行完畢 🎉\n" + "="*50)
for idx, res in enumerate(chat_results):
    print(f"\n--- 階段 {idx+1} 成果總結 ---")
    print(res.summary)


🎉 所有階段執行完畢 🎉

--- 階段 1 成果總結 ---
創業點子：智能知識庫管理平台

**概念概述：**
針對中小企業（SMEs），提供一個基於 RAG（Retrieval-Augmented Generation）技術的智能知識庫管理平台。此平台旨在解決中小企業在知識管理、信息檢索和員工培訓方面的痛點，幫助他們更高效地運營和創新。

**痛點分析：**
1. **知識管理混亂：** 中小企業通常缺乏系統化的知識管理工具，導致信息散落、重複和難以檢索。
2. **員工培訓成本高：** 員工流動性大，培訓新員工的時間和資源成本高昂。
3. **信息檢索效率低：** 員工花費大量時間在搜索和整理信息上，影響工作效率。

**解決方案：**
- **智能檢索：** 利用 RAG 技術，結合企業內部文件和外部數據源，提供準確且上下文相關的信息檢索。
- **自動化知識更新：** 平台自動從外部數據源更新相關行業資訊，保持知識庫的時效性。
- **個性化學習模組：** 根據員工角色和需求，提供定制化的學習資源和培訓模組。

**商業模式：**
- **訂閱制：** 按月或按年收取訂閱費用，根據企業規模和功能需求設計不同的訂閱層級。
- **增值服務：** 提供數據分析報告、專業顧問服務等增值服務，收取額外費用。

**市場機會：**
- **高需求：** 隨著數位化轉型的加速，中小企業對高效知識管理工具的需求日益增加。
- **低競爭：** 目前市場上針對中小企業的智能知識庫解決方案較少，具備先發優勢。

**高溢價空間分析：**
- **專業感：** 中小企業願意為提升內部運營效率和專業形象支付溢價。
- **氛圍感：** 提供一個現代化、智能化的工作環境，提升員工滿意度和歸屬感。
- **社群認同感：** 透過平台的使用，企業能夠更好地融入數位化商業生態，獲得行業認同。

**行動計劃：**
1. **產品開發：** 建立核心技術團隊，開發 MVP（最小可行產品）。
2. **市場測試：** 選擇特定行業進行試點，收集用戶反饋進行產品優化。
3. **市場擴展：** 擴大市場推廣，通過線上營銷和參加行業展會提升品牌知名度。

此創業點子旨在利用先進技術解決中小企業的實際問題，並通過靈活的商業模式實現盈利。

--- 階段 2 成果總結 ---
這個創業企劃看似